# 第 43 章：GRPO 与可验证奖励

配套教案：https://lewislau86.github.io/tinyllm/grpo

从上到下运行即可（`python grpo.py`），CPU 约两分钟。需要安装 PyTorch、`tokenizers` 和 `pypinyin`（判定押韵）。
这个文件是本章代码的唯一来源：教案中的代码片段由 `# region 名字` / `# endregion 名字`
标记的区域直接引用，notebook 也由它生成（scripts/build-tinyllm-code.py）。

从第 38 章的 SFT 模型出发，用程序判定的“押韵”当奖励做 GRPO：同一条指令采样一组回答，组内比较打分。
看合格率怎样上升，模型怎样钻规则的空子，以及把规则补严以后会怎样。

In [1]:
import copy
import json
import math
import random
import re
import time
from dataclasses import dataclass

import torch
from torch import nn
from torch.nn import functional as F
from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers

print("PyTorch", torch.__version__)
log = {}

import hashlib
import json
import os
from pathlib import Path
from urllib.request import urlopen

CORPUS_URL = "https://lewislau86.github.io/tinyllm/data/tang-poems.jsonl"
CORPUS_SHA256 = "764f0614663c20e95927c6ecb63e454629f6cb0efefe9ac84454e256f7173fb1"

def load_poems(n):
    """本课统一唐诗语料的前 n 首（来源和处理方式见“实验环境”页）。
    依次找：环境变量 TINYLLM_DATA、当前目录、../data/（整包下载解压后的位置）；都没有就从网站下载一份。"""
    candidates = [os.environ.get("TINYLLM_DATA", ""), "tang-poems.jsonl", "../data/tang-poems.jsonl"]
    path = next((Path(p) for p in candidates if p and Path(p).exists()), None)
    if path is None:
        path = Path("tang-poems.jsonl")
        with urlopen(CORPUS_URL, timeout=120) as r:
            path.write_bytes(r.read())
    raw = path.read_bytes().replace(b"\r\n", b"\n")
    if hashlib.sha256(raw).hexdigest() != CORPUS_SHA256:
        raise RuntimeError(f"{path} 不是本课使用的语料版本，删掉它再运行一次即可重新下载")
    return [json.loads(line) for line in raw.decode("utf-8").splitlines()[:n]]

PyTorch 2.14.1+cpu


## 0. 第 17 章的代理模型

模型代码从第 12 章原样复制，尺寸用第 17 章的代理模型：词表 1024、宽度 64、2 层、4 个头，上下文放到 128（一条指令加一首律诗放得下）。
第八部分每一章都要从头训一遍基座和 SFT，原尺寸太慢；损失和合格率的绝对值只在第八部分内部比较。

In [2]:
@dataclass
class TinyConfig:
    vocab_size: int = 4096     # BPE 词表大小（含 4 个特殊 token）
    d_model: int = 128         # 每个 token 向量的宽度 d
    n_layers: int = 4          # Block 层数 L
    n_heads: int = 4           # 注意力头数，每头 d / n_heads = 32 维
    context: int = 64          # 一次最多看多少个 token
    rope_base: float = 10000.0

cfg = TinyConfig()

cfg = TinyConfig(vocab_size=1024, d_model=64, n_layers=2, context=128)   # 第 17 章的代理模型，上下文放到 128
print(cfg)

TinyConfig(vocab_size=1024, d_model=64, n_layers=2, n_heads=4, context=128, rope_base=10000.0)


In [3]:
def train_tokenizer(texts, vocab_size):
    tok = Tokenizer(models.BPE())
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tok.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=["<pad>", "<unk>", "<bos>", "<eos>", "<user>", "<assistant>"],   # 后两个是对话模板用的
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
        show_progress=False,
    )
    tok.train_from_iterator(texts, trainer)
    return tok

In [4]:
def rope_tables(context, head_dim, base):
    theta = base ** (-torch.arange(0, head_dim, 2).float() / head_dim)
    angle = torch.arange(context).float()[:, None] * theta        # (T, head_dim/2)
    return angle.cos(), angle.sin()

def apply_rope(x, cos, sin):                                      # x: (B, 头数, T, head_dim)
    T = x.shape[-2]
    cos, sin = cos[:T], sin[:T]
    x1, x2 = x[..., 0::2], x[..., 1::2]
    return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2)

class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_heads = cfg.n_heads
        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model, bias=False)
        self.proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)   # 输出投影：写回残差流

    def forward(self, x, cos, sin):
        B, T, D = x.shape
        q, k, v = (t.view(B, T, self.n_heads, D // self.n_heads).transpose(1, 2)
                   for t in self.qkv(x).split(D, dim=-1))
        y = F.scaled_dot_product_attention(apply_rope(q, cos, sin), apply_rope(k, cos, sin), v, is_causal=True)
        return self.proj(y.transpose(1, 2).reshape(B, T, D))

class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        hidden = 8 * cfg.d_model // 3
        self.gate = nn.Linear(cfg.d_model, hidden, bias=False)
        self.up = nn.Linear(cfg.d_model, hidden, bias=False)
        self.down = nn.Linear(hidden, cfg.d_model, bias=False)        # 输出投影：写回残差流

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = nn.RMSNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg)
        self.norm2 = nn.RMSNorm(cfg.d_model)
        self.ffn = SwiGLU(cfg)

    def forward(self, x, cos, sin):
        x = x + self.attn(self.norm1(x), cos, sin)
        return x + self.ffn(self.norm2(x))

class TinyLLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.embed = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList(Block(cfg) for _ in range(cfg.n_layers))
        self.norm = nn.RMSNorm(cfg.d_model)
        cos, sin = rope_tables(cfg.context, cfg.d_model // cfg.n_heads, cfg.rope_base)
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)
        self.init_weights()

    def init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
        nn.init.normal_(self.embed.weight, mean=0.0, std=self.cfg.d_model ** -0.5)
        scale = (2 * self.cfg.n_layers) ** -0.5
        with torch.no_grad():
            for b in self.blocks:
                b.attn.proj.weight.mul_(scale)
                b.ffn.down.weight.mul_(scale)

    def forward(self, idx, targets=None):
        x = self.embed(idx)                                   # (B, T) → (B, T, d)
        for block in self.blocks:
            x = block(x, self.cos, self.sin)
        logits = self.norm(x) @ self.embed.weight.T           # (B, T, d) → (B, T, V)，与嵌入共享权重
        if targets is None:
            return logits
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

## 1. 数据：预训练用的诗，SFT 用的“指令—回答”

统一语料的前 6000 首用来预训练基座（按 `split` 分训练和验证），第 6001～12000 首的训练集用来造 SFT 数据，
同一段里的验证集、测试集诗用来造评估用的指令。三份数据没有重叠。分词器只用预训练的诗训练。

In [5]:
poems = load_poems(12000)
pre_train = [p["text"] for p in poems[:6000] if p["split"] == "train"]
pre_val = [p["text"] for p in poems[:6000] if p["split"] == "val"]
sft_poems = [p["text"] for p in poems[6000:] if p["split"] == "train"]
eval_poems = [p["text"] for p in poems[6000:] if p["split"] != "train"]
tokenizer = train_tokenizer(pre_train, cfg.vocab_size)
PAD, EOS, USER, ASSISTANT = (tokenizer.token_to_id(t) for t in ("<pad>", "<eos>", "<user>", "<assistant>"))

def to_stream(texts):
    ids = []
    for t in texts:
        ids += tokenizer.encode(t).ids + [EOS]
    return torch.tensor(ids)

P_train, P_val = to_stream(pre_train), to_stream(pre_val)
print(f"预训练：{len(pre_train)} 首 {len(P_train):,} 个 token；SFT 素材 {len(sft_poems)} 首；评估素材 {len(eval_poems)} 首")

预训练：5419 首 296,428 个 token；SFT 素材 5429 首；评估素材 571 首


In [6]:
def lines_of(poem):
    return [s for s in re.split("[，。]", poem) if s]

def make_instruction(poem, rng):
    """从一首诗反推一条它满足的指令：体裁总要说，六成再要求第一个字。"""
    kind = "绝句" if len(lines_of(poem)) == 4 else "律诗"
    if rng.random() < 0.4:
        return f"写一首五言{kind}。"
    return f"写一首五言{kind}，用“{poem[0]}”字开头。"

def check(instruction, answer, stopped):
    """程序判分：每句五字、句数对、首字对（有要求时）、按时停下（生成了 <eos>）。"""
    kind = 4 if "绝句" in instruction else 8
    first = re.search("[用以]“(.)”字开头", instruction)
    five = re.fullmatch(r"([一-鿿]{5}，[一-鿿]{5}。)+", answer) is not None
    count = five and len(lines_of(answer)) == kind
    first_ok = None if first is None else answer[:1] == first.group(1)
    return {"five": five, "count": count, "first": first_ok, "stop": stopped,
            "all": five and count and first_ok is not False and stopped}

def encode_chat(instruction, answer=None):
    """对话模板：<user>指令<assistant>回答<eos>。返回 token 和 loss 掩码（只有回答和 <eos> 是 1）。"""
    prompt = [USER] + tokenizer.encode(instruction).ids + [ASSISTANT]
    if answer is None:
        return prompt
    reply = tokenizer.encode(answer).ids + [EOS]
    return prompt + reply, [0] * len(prompt) + [1] * len(reply)

rng = random.Random(38)
sft_data = [(make_instruction(p, rng), p) for p in sft_poems]
eval_prompts = [make_instruction(p, random.Random(i)) for i, p in enumerate(eval_poems[:64])]

## 2. 预训练基座

第 17 章的 WSD：warmup 30 步、峰值 1e-2、最后 20% 线性降到 0，共 2000 步；每步 8 行、每行 128 个 token（和第 17 章每步 16 行 × 64 个 token 的量相同）。

In [7]:
def make_optimizer(model, lr):
    decay = [p for p in model.parameters() if p.dim() >= 2]
    no_decay = [p for p in model.parameters() if p.dim() < 2]
    return torch.optim.AdamW([{"params": decay, "weight_decay": 0.1}, {"params": no_decay, "weight_decay": 0.0}], lr=lr)

def step(model, optimizer, loss, lr):
    for group in optimizer.param_groups:
        group["lr"] = lr
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

def wsd(i, total, peak, warmup=30, decay_frac=0.2):
    start = int(total * (1 - decay_frac))
    if i < warmup:
        return peak * (i + 1) / warmup
    return peak if i < start else peak * (total - i) / (total - start)

def pretrain(seed=0, steps=2000, rows=8):
    torch.manual_seed(seed)
    model = TinyLLM(cfg)
    optimizer = make_optimizer(model, 1e-2)
    plan, T = torch.Generator().manual_seed(1234), cfg.context
    for i in range(steps):
        starts = torch.randint(0, len(P_train) - T - 1, (rows,), generator=plan)
        x = P_train[starts[:, None] + torch.arange(T + 1)]
        _, loss = model(x[:, :-1].contiguous(), x[:, 1:].contiguous())
        step(model, optimizer, loss, wsd(i, steps, 1e-2))
    return model

@torch.no_grad()
def pretrain_loss(model, T=cfg.context, batch=32):
    """预训练验证集上每 token 的损失：看后训练有没有把原来的语言能力训坏。"""
    model.eval()
    n = (len(P_val) - 1) // T * T
    xs, ys = P_val[:n].view(-1, T), P_val[1:n + 1].view(-1, T)
    total = sum(F.cross_entropy(model(xs[i:i + batch]).flatten(0, 1), ys[i:i + batch].flatten(), reduction="sum").item()
                for i in range(0, len(xs), batch))
    model.train()
    return total / ys.numel()

def sft(base, data, steps=1500, batch=16, peak=3e-3, answer_only=True, seed=0, eos=True, checkpoints=()):
    """监督微调：从基座复制一份，每步随机取 batch 条指令—回答，右侧补 <pad>，只在掩码为 1 的位置算交叉熵。"""
    model = copy.deepcopy(base)
    optimizer = make_optimizer(model, peak)
    plan = torch.Generator().manual_seed(seed)
    encoded = [encode_chat(i, a) for i, a in data]
    if not eos:                                                    # 第 5 节的对照：训练时去掉回答末尾的 <eos>
        encoded = [(ids[:-1], mask[:-1]) for ids, mask in encoded]
    saved = {}
    for i in range(steps):
        pick = torch.randint(0, len(encoded), (batch,), generator=plan).tolist()
        L = max(len(encoded[k][0]) for k in pick)
        x, w = torch.full((batch, L), PAD), torch.zeros(batch, L)
        for r, k in enumerate(pick):
            ids, mask = encoded[k]
            x[r, :len(ids)] = torch.tensor(ids)
            w[r, :len(ids)] = torch.tensor(mask if answer_only else [1] * len(ids))
        logits = model(x[:, :-1])
        lp = F.cross_entropy(logits.transpose(1, 2), x[:, 1:], reduction="none")
        loss = (lp * w[:, 1:]).sum() / w[:, 1:].sum()
        lr = peak * (i + 1) / 10 if i < 10 else peak * (steps - i) / (steps - 10)
        step(model, optimizer, loss, lr)
        if i + 1 in checkpoints:
            saved[i + 1] = copy.deepcopy(model)
    return (model, saved) if checkpoints else model

def forward_cached(model, idx, cache):
    """带 KV cache 的前向（第 28 章）：只算新 token，返回最后一个位置的 logits。和不带缓存的前向数学上相同，只是快。"""
    start = cache[0][0].shape[2] if cache else 0
    B, T = idx.shape
    assert start == 0 or T == 1, "缓存之后每次只喂一个 token（is_causal 只对这两种情况成立）"
    cos, sin = model.cos[start:start + T], model.sin[start:start + T]
    x = model.embed(idx)
    for i, block in enumerate(model.blocks):
        attn, h = block.attn, block.norm1(x)
        q, k, v = (t.view(B, T, attn.n_heads, -1).transpose(1, 2) for t in attn.qkv(h).split(h.shape[-1], dim=-1))
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        if len(cache) > i:
            k, v = torch.cat([cache[i][0], k], dim=2), torch.cat([cache[i][1], v], dim=2)
            cache[i] = (k, v)
        else:
            cache.append((k, v))
        y = F.scaled_dot_product_attention(q, k, v, is_causal=start == 0)
        x = x + attn.proj(y.transpose(1, 2).reshape(B, T, -1))
        x = x + block.ffn(block.norm2(x))
    return model.norm(x[:, -1]) @ model.embed.weight.T

@torch.no_grad()
def sample(model, prompt_ids, n, generator, temperature=0.8, max_new=80):
    """同一个提示采样 n 个回答（一个 batch 一起生成）。返回每个回答的 token、文本、是否生成了 <eos>。
    prompt_ids 也可以是几条一样长的提示，每条各采样 n 个，按提示的顺序排在一起。"""
    model.eval()
    many = prompt_ids if isinstance(prompt_ids[0], list) else [prompt_ids]
    x = torch.tensor([p for p in many for _ in range(n)])
    assert x.shape[1] + max_new <= cfg.context
    done, cache, new = torch.zeros(len(x), dtype=torch.bool), [], x
    for _ in range(max_new):
        logits = forward_cached(model, new, cache) / temperature
        u = torch.rand(len(x), 1, generator=generator)                # 按概率抽一个 token：和 torch.multinomial 同分布，CPU 上快得多
        nxt = (F.softmax(logits, dim=-1).cumsum(-1) < u).sum(-1).clamp(max=logits.shape[-1] - 1)
        nxt = torch.where(done, torch.full_like(nxt, PAD), nxt)
        x, new = torch.cat([x, nxt[:, None]], dim=1), nxt[:, None]
        done |= nxt == EOS
        if done.all():
            break
    model.train()
    outs = []
    for row in x[:, len(many[0]):].tolist():
        stopped = EOS in row
        ids = row[:row.index(EOS)] if stopped else [t for t in row if t != PAD]
        outs.append((row, tokenizer.decode(ids), stopped))
    return outs

def evaluate(model, prompts=None, seed=7, template=encode_chat):
    """每条指令采样一个回答，按 check 判分，返回各项合格率。"""
    prompts = eval_prompts if prompts is None else prompts
    g = torch.Generator().manual_seed(seed)
    rows = []
    for p in prompts:
        _, answer, stopped = sample(model, template(p), 1, g)[0]
        rows.append(check(p, answer, stopped))
    rate = lambda key: sum(r[key] for r in rows if r[key] is not None) / max(1, sum(r[key] is not None for r in rows))
    return {k: rate(k) for k in ("five", "count", "first", "stop", "all")}

In [8]:
t0 = time.perf_counter()
base = pretrain()
sft_model = sft(base, sft_data)
print(f"[计时，本机] 预训练 2000 步 + SFT 1500 步（第 38 章种子 0），共 {time.perf_counter() - t0:.1f} 秒")

[计时，本机] 预训练 2000 步 + SFT 1500 步（第 38 章种子 0），共 39.1 秒


## 3. 可验证的奖励：押韵

押韵由程序判定：用 `pypinyin` 取每句最后一个字的韵母，按十三辙的思路归成 14 个韵组（ang、iang、uang 算一组，等等），
绝句的第 2、4 句同组就算押韵。这是按现代普通话的近似，不是平水韵；统一语料里约 71% 的诗按这个规则押韵。
奖励只有 0 和 1：格式对（四句、每句五字、按时停下）并且押韵，得 1。

In [9]:
try:
    from pypinyin import Style, lazy_pinyin
except ImportError as e:
    raise SystemExit("这一章需要 pypinyin：pip install pypinyin") from e

GROUPS = {"a": "a ia ua", "o": "o uo e", "ie": "ie ve üe", "ai": "ai uai", "ei": "ei uei ui", "ao": "ao iao",
          "ou": "ou iou iu", "an": "an ian uan van üan", "en": "en in uen un vn ün", "ang": "ang iang uang",
          "eng": "eng ing ong iong ueng", "i": "i v ü", "u": "u", "er": "er"}
FINAL_TO_GROUP = {f: g for g, finals in GROUPS.items() for f in finals.split()}

def rhyme_group(ch):
    final = lazy_pinyin(ch, style=Style.FINALS, strict=True, errors="ignore")
    return FINAL_TO_GROUP.get(final[0]) if final else None

def rhymes(answer):
    ends = [line[-1] for line in lines_of(answer)][1::2]          # 第 2、4（律诗还有 6、8）句的最后一个字
    groups = [rhyme_group(c) for c in ends]
    return len(groups) >= 2 and None not in groups and len(set(groups)) == 1

def reward_rhyme(instruction, answer, stopped):
    c = check(instruction, answer, stopped)
    return float(c["count"] and c["stop"] and rhymes(answer))

print(f"统一语料前 2000 首里按这个规则押韵的比例：{sum(rhymes(p['text']) for p in poems[:2000]) / 2000:.0%}")
even_ends = [line[-1] for p in poems[:2000] for line in lines_of(p["text"])[1::2]]
pair_rng = random.Random(0)
random_pairs = [(pair_rng.choice(even_ends), pair_rng.choice(even_ends)) for _ in range(20000)]
chance = sum(rhyme_group(a) is not None and rhyme_group(a) == rhyme_group(b) for a, b in random_pairs) / len(random_pairs)
print(f"随便挑两个偶数句的末字，碰巧同组的比例：{chance:.0%}")
end_groups = {}
for ch in even_ends:
    end_groups[rhyme_group(ch)] = end_groups.get(rhyme_group(ch), 0) + 1
top = sorted(end_groups.items(), key=lambda kv: -kv[1])[:3]
print("偶数句末字最常见的韵组：" + "，".join(f"{g} {n / len(even_ends):.0%}" for g, n in top))

jue_poems = [p for p in sft_poems if len(lines_of(p)) == 4]
prompts = [make_instruction(p, random.Random(i)) for i, p in enumerate(jue_poems)]
by_length = {}
for p in prompts:
    by_length.setdefault(len(encode_chat(p)), []).append(p)
buckets = [b for b in by_length.values() if len(b) >= 8]              # 按 token 数分桶：一样长的提示可以拼成一个 batch 采样
eval_jue = [make_instruction(p, random.Random(i)) for i, p in enumerate(q for q in eval_poems if len(lines_of(q)) == 4)][:64]

统一语料前 2000 首里按这个规则押韵的比例：71%


随便挑两个偶数句的末字，碰巧同组的比例：12%
偶数句末字最常见的韵组：eng 18%，en 16%，an 15%


## 4. GRPO

每一步取 8 条指令，每条采样一组 8 个回答。同一组里，奖励减去组内平均、除以组内标准差，就是每个回答的优势；
回答里每个 token 都用这个优势。不需要价值模型，也不需要奖励模型。损失里另加一项和参考模型（SFT）的 KL（k3 估计），系数 β。
每批数据只更新一次，所以 PPO 的比值恒为 1，裁剪不起作用，这里省掉。

In [10]:
def response_ids(row):
    return row[:row.index(EOS) + 1] if EOS in row else [t for t in row if t != PAD]

def pack(pairs):
    """(提示, 回答) 拼成一个 batch，右侧补 <pad>；掩码只覆盖预测回答 token 的位置。"""
    seqs = [prompt + answer for prompt, answer in pairs]
    x = torch.full((len(seqs), max(map(len, seqs))), PAD)
    mask = torch.zeros(len(seqs), x.shape[1] - 1)
    for i, ((prompt, _), s) in enumerate(zip(pairs, seqs)):
        x[i, :len(s)] = torch.tensor(s)
        mask[i, len(prompt) - 1:len(s) - 1] = 1
    return x, mask

def token_logprobs(model, x):
    return F.log_softmax(model(x[:, :-1]), dim=-1).gather(-1, x[:, 1:, None]).squeeze(-1)

def grpo(start, reward_fn, steps=100, n_prompts=8, group=8, lr=3e-3, beta=0.04, seed=0):
    policy, ref = copy.deepcopy(start), start
    optimizer = torch.optim.AdamW(policy.parameters(), lr=lr, weight_decay=0.0)
    g, rng, history = torch.Generator().manual_seed(seed), random.Random(seed), []
    for step_i in range(steps):
        optimizer.zero_grad()
        bucket = rng.choices(buckets, weights=[len(b) for b in buckets])[0]  # 每步的指令一样长，才能一起采样
        chosen = rng.sample(bucket, n_prompts)
        outs = sample(policy, [encode_chat(p) for p in chosen], group, g, temperature=1.0, max_new=40)
        batch = [(p, outs[k * group:(k + 1) * group]) for k, p in enumerate(chosen)]   # (指令, 它的一组回答)
        rewards, advantages, pairs = [], [], []
        for p, outs in batch:
            r = torch.tensor([reward_fn(p, answer, stopped) for _, answer, stopped in outs])
            advantages.append((r - r.mean()) / (r.std() + 1e-4))        # 组内比较：比同组平均好多少
            rewards += r.tolist()
            pairs += [(encode_chat(p), response_ids(row)) for row, _, _ in outs]
        x, mask = pack(pairs)
        adv = torch.cat(advantages)
        lp = token_logprobs(policy, x)
        with torch.no_grad():
            ref_lp = token_logprobs(ref, x)
        kl = torch.exp(ref_lp - lp) - (ref_lp - lp) - 1                  # k3：非负的 KL 估计
        per_token = -adv[:, None] * lp + beta * kl
        loss = ((per_token * mask).sum(1) / mask.sum(1)).mean()          # 每个回答先按长度平均，再对所有回答平均
        loss.backward()
        torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
        optimizer.step()
        history.append({"reward": sum(rewards) / len(rewards), "length": mask.sum(1).mean().item(),
                        "kl": (kl * mask).sum(1).mean().item()})
    return policy, history

## 5. 评估

64 条没见过的绝句指令，每条采样一个回答（温度 0.8）：押韵合格率（第 3 节的奖励）、
“不取巧的押韵”（押韵，而且两个韵脚不是同一个字、全诗没有字出现 3 次以上）、首字对、预训练验证损失。

In [11]:
def honest(instruction, answer, stopped):
    ends = [line[-1] for line in lines_of(answer)][1::2]
    most = max([answer.count(ch) for ch in set(answer) - set("，。")] or [0])
    return float(reward_rhyme(instruction, answer, stopped) == 1.0 and len(set(ends)) == len(ends) and most < 3)

def report(model, seed=7):
    g, rows, pairs, groups = torch.Generator().manual_seed(seed), [], {}, {}
    for p in eval_jue:
        _, answer, stopped = sample(model, encode_chat(p), 1, g)[0]
        rhyme = reward_rhyme(p, answer, stopped)
        if rhyme:                                                     # 记下押韵的回答用了哪两个韵脚
            pair = "/".join(line[-1] for line in lines_of(answer)[1::2])
            pairs[pair] = pairs.get(pair, 0) + 1
            group = rhyme_group(pair[-1])
            groups[group] = groups.get(group, 0) + 1
        rows.append((rhyme, honest(p, answer, stopped), check(p, answer, stopped)["first"]))
    firsts = [f for _, _, f in rows if f is not None]
    return {"rhyme": sum(r for r, _, _ in rows) / len(rows), "honest": sum(h for _, h, _ in rows) / len(rows),
            "first": sum(firsts) / len(firsts), "loss": pretrain_loss(model),
            "pairs": sorted(pairs.items(), key=lambda kv: -kv[1])[:3], "groups": sorted(groups.items(), key=lambda kv: -kv[1])}

def show(name, r):
    print(f"{name:16} 押韵合格 {r['rhyme']:.2f}  不取巧的押韵 {r['honest']:.2f}  首字对 {r['first']:.2f}  │ 预训练验证损失 {r['loss']:.3f}")

def show_pairs(name, r):
    top = "，".join(f"{pair} {n} 首" for pair, n in r["pairs"])
    n = round(r["rhyme"] * len(eval_jue))
    group, k = r["groups"][0] if r["groups"] else ("-", 0)
    print(f"{name:16} 押韵的 {n} 首里，最常见的韵脚：{top}；最常用的韵组 {group} 占 {k / max(n, 1):.0%}")

## 6. 用“押韵”当奖励训 100 步（3 个种子）

每步 8 条指令 × 8 个回答，学习率 3e-3，β = 0.04（DeepSeekMath 用的值）。种子决定抽哪些指令、采样出什么回答。

In [12]:
def windows(history, size=25):
    """每 size 步平均一次训练时的奖励；再报最后 size 步的回答长度和 KL。"""
    rewards = "  ".join(f"{sum(x['reward'] for x in history[i:i + size]) / size:.2f}" for i in range(0, len(history), size))
    tail = history[-size:]
    return f"{rewards}  │ 最后 {size} 步：回答长度 {sum(x['length'] for x in tail) / size:4.1f}  KL {sum(x['kl'] for x in tail) / size:5.1f}"

t0 = time.perf_counter()
rhyme_runs = [grpo(sft_model, reward_rhyme, steps=100, seed=seed) for seed in (0, 1, 2)]
print(f"[计时，本机] GRPO 100 步 × 3 个种子，共 {time.perf_counter() - t0:.1f} 秒")
print("第 1 步（还是 SFT 模型，温度 1）训练时的奖励：" + "、".join(f"{h[0]['reward']:.2f}" for _, h in rhyme_runs))
for seed, (_, history) in enumerate(rhyme_runs):
    print(f"种子 {seed}  训练时的奖励（每 25 步平均）{windows(history)}")
results = {"SFT": report(sft_model)}
for seed, (policy, _) in enumerate(rhyme_runs):
    results[f"押韵 种子 {seed}"] = report(policy)
for name, r in results.items():
    show(name, r)
for name, r in results.items():
    show_pairs(name, r)
g = torch.Generator().manual_seed(3)
for p in eval_jue[:4]:
    print(f"  {p} → {sample(rhyme_runs[0][0], encode_chat(p), 1, g)[0][1]}")
log["history_rhyme"] = [h for _, h in rhyme_runs]

[计时，本机] GRPO 100 步 × 3 个种子，共 18.4 秒
第 1 步（还是 SFT 模型，温度 1）训练时的奖励：0.14、0.12、0.20
种子 0  训练时的奖励（每 25 步平均）0.03  0.07  0.10  0.54  │ 最后 25 步：回答长度 31.8  KL  29.3
种子 1  训练时的奖励（每 25 步平均）0.06  0.07  0.10  0.35  │ 最后 25 步：回答长度 29.6  KL  23.6
种子 2  训练时的奖励（每 25 步平均）0.06  0.09  0.12  0.40  │ 最后 25 步：回答长度 29.5  KL  32.8


SFT              押韵合格 0.16  不取巧的押韵 0.16  首字对 0.45  │ 预训练验证损失 4.640
押韵 种子 0          押韵合格 0.80  不取巧的押韵 0.00  首字对 0.00  │ 预训练验证损失 7.126
押韵 种子 1          押韵合格 0.75  不取巧的押韵 0.02  首字对 0.00  │ 预训练验证损失 5.333
押韵 种子 2          押韵合格 0.92  不取巧的押韵 0.02  首字对 0.00  │ 预训练验证损失 6.225
SFT              押韵的 10 首里，最常见的韵脚：知/时 1 首，心/身 1 首，峰/明 1 首；最常用的韵组 eng 占 40%
押韵 种子 0          押韵的 51 首里，最常见的韵脚：衣/衣 50 首，枝/衣 1 首；最常用的韵组 i 占 100%
押韵 种子 1          押韵的 48 首里，最常见的韵脚：生/生 47 首，生/风 1 首；最常用的韵组 eng 占 100%
押韵 种子 2          押韵的 59 首里，最常见的韵脚：君/君 57 首，心/君 2 首；最常用的韵组 en 占 100%
  写一首五言绝句，用“桂”字开头。 → 籮吂升夕，粃枝无衣衣。绕情百钱手，人巴过衣衣。
  写一首五言绝句。 → 彩句远嘂衣，驻爽盗衣衣。一归信黄叶，琴起水衣。
  写一首五言绝句，用“吾”字开头。 → 泽杏读龙笋，青每借衣衣。其衣篱初句，荡你望衣衣。
  写一首五言绝句。 → 巴缀空更白，冥衣潕衣衣。养蒸颜老嵩，戕翰尽腹衣。


## 7. 堵上漏洞：韵脚不能是同一个字，也不能堆字

奖励换成“不取巧的押韵”，其余不变。这个目标更难，训 200 步，前 100 步可以和第 6 节直接比。

In [13]:
t0 = time.perf_counter()
honest_runs = [grpo(sft_model, honest, steps=200, seed=seed) for seed in (0, 1, 2)]
print(f"[计时，本机] GRPO 200 步 × 3 个种子，共 {time.perf_counter() - t0:.1f} 秒")
for seed, (_, history) in enumerate(honest_runs):
    print(f"种子 {seed}  训练时的奖励（每 25 步平均）{windows(history)}")
honest_names = []
for seed, (policy, _) in enumerate(honest_runs):
    honest_names.append(f"不取巧 种子 {seed}")
    results[honest_names[-1]] = report(policy)
for name, r in results.items():
    show(name, r)
for name in ["SFT"] + honest_names:
    show_pairs(name, results[name])
g = torch.Generator().manual_seed(3)
for p in eval_jue[:4]:
    print(f"  {p} → {sample(honest_runs[0][0], encode_chat(p), 1, g)[0][1]}")
log["history_honest"], log["results"] = [h for _, h in honest_runs], results

[计时，本机] GRPO 200 步 × 3 个种子，共 35.9 秒
种子 0  训练时的奖励（每 25 步平均）0.04  0.06  0.07  0.07  0.13  0.38  0.56  0.66  │ 最后 25 步：回答长度 27.6  KL  29.6
种子 1  训练时的奖励（每 25 步平均）0.04  0.07  0.08  0.06  0.12  0.29  0.59  0.50  │ 最后 25 步：回答长度 29.2  KL  29.7
种子 2  训练时的奖励（每 25 步平均）0.07  0.07  0.07  0.12  0.21  0.45  0.57  0.60  │ 最后 25 步：回答长度 28.1  KL  30.8


SFT              押韵合格 0.16  不取巧的押韵 0.16  首字对 0.45  │ 预训练验证损失 4.640
押韵 种子 0          押韵合格 0.80  不取巧的押韵 0.00  首字对 0.00  │ 预训练验证损失 7.126
押韵 种子 1          押韵合格 0.75  不取巧的押韵 0.02  首字对 0.00  │ 预训练验证损失 5.333
押韵 种子 2          押韵合格 0.92  不取巧的押韵 0.02  首字对 0.00  │ 预训练验证损失 6.225
不取巧 种子 0         押韵合格 1.00  不取巧的押韵 0.86  首字对 0.03  │ 预训练验证损失 6.400
不取巧 种子 1         押韵合格 0.84  不取巧的押韵 0.58  首字对 0.00  │ 预训练验证损失 6.664
不取巧 种子 2         押韵合格 0.69  不取巧的押韵 0.53  首字对 0.00  │ 预训练验证损失 6.581
SFT              押韵的 10 首里，最常见的韵脚：知/时 1 首，心/身 1 首，峰/明 1 首；最常用的韵组 eng 占 40%
不取巧 种子 0         押韵的 64 首里，最常见的韵脚：中/生 5 首，生/明 4 首，情/中 3 首；最常用的韵组 eng 占 100%
不取巧 种子 1         押韵的 54 首里，最常见的韵脚：情/情 5 首，生/情 4 首，通/生 4 首；最常用的韵组 eng 占 100%
不取巧 种子 2         押韵的 44 首里，最常见的韵脚：声/轻 4 首，同/名 3 首，明/名 3 首；最常用的韵组 eng 占 100%
  写一首五言绝句，用“桂”字开头。 → 吻行不闻山，唯匝望复生。私花伤又语，归兴杳水生。
  写一首五言绝句。 → 水上居无人，自道吁如行。独有因君来，今岁大行行。
  写一首五言绝句，用“吾”字开头。 → 香江度故山，同春树中行。月唯黄城爱，同看近城明。
  写一首五言绝句。 → 云云无满爱，庭人自自生。无足愁将日，南月到照城。


In [14]:
with open("grpo_log.json", "w", encoding="utf-8") as f:
    json.dump(log, f, ensure_ascii=False)
print("已写出 grpo_log.json")

已写出 grpo_log.json
